In [ ]:
import os
import pandas as pd

def count_lines(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

old_train_data = pd.read_csv('Data/drug_review_train.csv')
train_data = pd.read_csv('Data/drug_review_train_clean.csv')
val_data = pd.read_csv('Data/drug_review_validation_clean.csv')
test_data = pd.read_csv('Data/drug_review_test_clean.csv')

#Define the features (X) and the target variable (y)
label = "rating"
X_train = train_data.drop(columns=[label])  # Drop the target column
y_train = train_data[label]  # Target column
X_val = val_data.drop(columns=[label])  # Drop the target column
y_val = val_data[label]  # Target columnX_train = train_data.drop(columns=[label])  # Drop the target column
X_test = test_data.drop(columns=[label])  # Drop the target column
y_test = test_data[label]  # Target column

old_train_data.head(10)
train_data.head(10)


,Unnamed: 0,patient_id,drugName,condition,review,rating,date,usefulCount,review_length,review_clean
0,0,89879,Cyclosporine,keratoconjunctivitis sicca,"""i have used restasis for about a year now and...",2.0,"April 20, 2013",69,147,restasis year see progress life red bothersome...
1,1,143975,Etonogestrel,birth control,"""my experience has been somewhat mixed. i have...",7.0,"August 7, 2016",4,136,experience somewhat mixed implanon nearly mont...
2,2,106473,Implanon,birth control,"""this is my second implanon would not recommen...",1.0,"May 11, 2016",6,140,implanon recommend allfirst okay year start bl...
3,3,184526,Hydroxyzine,anxiety,"""i recommend taking as prescribed, and the bot...",10.0,"March 19, 2012",124,104,recommend take prescribe bottle usually say ho...
4,4,91587,Dalfampridine,multiple sclerosis,"""i have been on ampyra for 5 days and have bee...",9.0,"August 1, 2010",101,74,ampyra day happy new pill day good effect day ...
5,5,218554,Tri-Sprintec,birth control,"""used for birth control and period issues- ver...",2.0,"January 7, 2017",4,57,birth control period unhappy work good prevent...
6,6,207442,Suprep Bowel Prep Kit,bowel preparation,"""my prep instructions were one 6oz bottle the ...",8.0,"December 5, 2016",18,86,prep instruction oz bottle evening oz bottle m...
7,7,63753,Epiduo,acne,"""love it. i had the worst breakouts, so i got ...",9.0,"October 19, 2011",20,63,love bad breakout get epiduo time show result ...
8,8,140845,Escitalopram,depression,"""i felt a positive difference within the first...",9.0,"April 21, 2016",13,49,feel positive difference week generalize anxie...
9,9,182520,Mirena,birth control,"""i have been on mirena for over a year now and...",1.0,"June 15, 2009",45,37,mirena year experience effect effect watch sev...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Get the text column (update if your text column has a different name)
text_column = 'review'

# Fill missing values
X_train[text_column] = X_train[text_column].fillna("")
X_val[text_column] = X_val[text_column].fillna("")
X_test[text_column] = X_test[text_column].fillna("")

# Initialize TF-IDF
tfidf = TfidfVectorizer(max_features=5000)  # You can tune this

# Fit TF-IDF on training data and transform all sets
X_train_tfidf = tfidf.fit_transform(X_train[text_column])
X_val_tfidf = tfidf.transform(X_val[text_column])
X_test_tfidf = tfidf.transform(X_test[text_column])

In [ ]:
def map_sentiment(rating):
    if rating >= 7:
        return 'positive'
    elif rating >= 4:
        return 'neutral'
    else:
        return 'negative'

y_train_sent = y_train.apply(map_sentiment)
y_val_sent = y_val.apply(map_sentiment)
y_test_sent = y_test.apply(map_sentiment)

# Encode strings into integers
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_sent)
y_val_enc = le.transform(y_val_sent)
y_test_enc = le.transform(y_test_sent)


In [ ]:
import lightgbm as lgb
from sklearn.feature_selection import SelectFromModel

# Train LWGBM
lgb_model = lgb.LGBMClassifier(n_estimators=100)
lgb_model.fit(X_train_tfidf, y_train_enc)

# Select features based on importance
selector = SelectFromModel(lgb_model, prefit=True, threshold='median')  # keep top 50%
X_train_sel = selector.transform(X_train_tfidf)
X_val_sel = selector.transform(X_val_tfidf)
X_test_sel = selector.transform(X_test_tfidf)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.799254 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 567749
[LightGBM] [Info] Number of data points in the train set: 110811, number of used features: 5000
[LightGBM] [Info] Start training from score -1.546495
[LightGBM] [Info] Start training from score -2.099891
[LightGBM] [Info] Start training from score -0.408665


In [ ]:
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import classification_report
import numpy as np

# Start H2O
h2o.init()

# Convert to H2OFrame
train_h2o = h2o.H2OFrame(pd.DataFrame(X_train_sel.toarray()))
train_h2o['label'] = h2o.H2OFrame(y_train_enc.astype('int'))

val_h2o = h2o.H2OFrame(pd.DataFrame(X_val_sel.toarray()))
val_h2o['label'] = h2o.H2OFrame(y_val_enc.astype('int'))

# Run AutoML
aml = H2OAutoML(max_models=10, seed=1)
aml.train(x=train_h2o.col_names[:-1], y='label', training_frame=train_h2o)

# Evaluate on validation set
preds = aml.predict(val_h2o).as_data_frame()['predict']
print(classification_report(y_val_enc, preds.astype(int), target_names=le.classes_))

# Shut down H2O (optional at the end)
# h2o.shutdown(prompt=False)


ModuleNotFoundError: No module named 'h2o'